In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "ecom_poc"

# ==========================================
# 1. ORDERS: VALIDATE -> QUARANTINE -> DEDUP
# ==========================================
b_orders = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")

orders_staged = (
    b_orders
    .withColumn("order_id_c", F.upper(F.trim(F.col("order_id"))))
    .withColumn("customer_id_c", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("order_date_c", F.expr("try_cast(trim(order_date) as date)"))
    .withColumn("category_c", F.initcap(F.trim(F.col("category"))))
    .withColumn("quantity_c", F.expr("try_cast(trim(quantity) as int)"))
    .withColumn("unit_price_c", F.expr("try_cast(trim(unit_price) as decimal(12,2))"))
    .withColumn("order_status_c", F.upper(F.trim(F.col("order_status"))))
    .withColumn("country_c", F.upper(F.trim(F.col("country"))))
)

orders_ruled = orders_staged.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("order_id_c").isNull() | (F.col("order_id_c") == "") |
            F.col("customer_id_c").isNull() | (F.col("customer_id_c") == "") |
            F.col("category_c").isNull() | (F.col("category_c") == "") |
            F.col("country_c").isNull() | (F.col("country_c") == ""),
            F.lit("MISSING_MANDATORY_FIELD")
        ),
        F.when(
            F.col("quantity_c").isNull() | F.col("unit_price_c").isNull(),
            F.lit("INVALID_NUMERIC_TYPE")
        ),
        F.when(
            (F.col("quantity_c").isNotNull() & ((F.col("quantity_c") <= 0) | (F.col("quantity_c") > 1000))) |
            (F.col("unit_price_c").isNotNull() & ((F.col("unit_price_c") <= 0) | (F.col("unit_price_c") > 100000))),
            F.lit("OUT_OF_RANGE_NUMERIC")
        ),
        F.when(
            F.col("order_date_c").isNull() | (F.col("order_date_c") > F.current_date()),
            F.lit("INVALID_OR_FUTURE_DATE")
        ),
        F.when(
            (~F.col("order_status_c").isin("COMPLETED", "SHIPPED", "PROCESSING", "CANCELLED")) |
            (~F.col("country_c").isin("US", "UK", "CA", "DE", "AU", "IN")),
            F.lit("INVALID_DOMAIN_LABEL")
        )
    )
)

# Quarantine invalid orders
orders_quarantine = orders_ruled.filter(F.col("rejection_reason") != "").select(
    "order_id", "customer_id", "order_date", "category", "quantity", "unit_price",
    "order_status", "country", "rejection_reason",
    "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
)
orders_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.quarantine_orders")

# Deduplicate valid orders deterministically
w_orders = Window.partitionBy("order_id_c").orderBy(
    F.col("order_date_c").desc(), F.col("_row_hash").asc(), F.col("_bronze_record_id").asc()
)

orders_valid_ranked = (
    orders_ruled.filter(F.col("rejection_reason") == "")
    .withColumn("_rn", F.row_number().over(w_orders))
    .withColumn("_winner_hash", F.first("_row_hash").over(w_orders))
)

silver_orders = orders_valid_ranked.filter(F.col("_rn") == 1).select(
    F.col("order_id_c").alias("order_id"),
    F.col("customer_id_c").alias("customer_id"),
    F.col("order_date_c").alias("order_date"),
    F.col("category_c").alias("category"),
    F.col("quantity_c").alias("quantity"),
    F.col("unit_price_c").alias("unit_price"),
    (F.col("quantity_c") * F.col("unit_price_c")).cast("decimal(14,2)").alias("order_amount"),
    F.col("order_status_c").alias("order_status"),
    F.col("country_c").alias("country"),
    "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
)
silver_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")

dup_orders = (
    orders_valid_ranked.filter(F.col("_rn") > 1)
    .withColumn(
        "duplicate_type",
        F.when(F.col("_row_hash") == F.col("_winner_hash"), F.lit("EXACT_ROW_DUPLICATE"))
         .otherwise(F.lit("BUSINESS_KEY_DUPLICATE"))
    )
    .select(
        F.col("order_id_c").alias("order_id"), "customer_id", "order_date", "category",
        "quantity", "unit_price", "order_status", "country", "duplicate_type",
        "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
    )
)
dup_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.duplicate_losers_orders")


# ================================================
# 2. TRANSACTIONS: VALIDATE -> QUARANTINE -> DEDUP
# ================================================
b_txns = spark.table(f"{CATALOG}.{SCHEMA}.bronze_transactions")
valid_order_keys = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders").select(F.col("order_id").alias("_ref_order_id")).distinct()

txns_staged = (
    b_txns
    .withColumn("txn_id_c", F.upper(F.trim(F.col("txn_id"))))
    .withColumn("order_id_c", F.upper(F.trim(F.col("order_id"))))
    .withColumn("payment_method_c", F.upper(F.trim(F.col("payment_method"))))
    .withColumn("amount_c", F.expr("try_cast(trim(amount) as decimal(12,2))"))
    .withColumn("currency_c", F.upper(F.trim(F.col("currency"))))
    .withColumn("txn_status_c", F.upper(F.trim(F.col("txn_status"))))
    .withColumn("txn_ts_c", F.expr("try_cast(trim(txn_ts) as timestamp)"))
    .join(valid_order_keys, F.col("order_id_c") == F.col("_ref_order_id"), "left")
)

txns_ruled = txns_staged.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("txn_id_c").isNull() | (F.col("txn_id_c") == "") |
            F.col("order_id_c").isNull() | (F.col("order_id_c") == "") |
            F.col("payment_method_c").isNull() | (F.col("payment_method_c") == ""),
            F.lit("MISSING_MANDATORY_FIELD")
        ),
        F.when(F.col("amount_c").isNull(), F.lit("INVALID_NUMERIC_TYPE")),
        F.when(
            F.col("amount_c").isNotNull() & ((F.col("amount_c") <= 0) | (F.col("amount_c") > 1000000)),
            F.lit("OUT_OF_RANGE_AMOUNT")
        ),
        F.when(
            F.col("txn_ts_c").isNull() | (F.col("txn_ts_c") > F.current_timestamp()),
            F.lit("INVALID_OR_FUTURE_TIMESTAMP")
        ),
        F.when(
            (~F.col("payment_method_c").isin("CARD", "PAYPAL", "BANK_TRANSFER", "APPLE_PAY")) |
            (~F.col("currency_c").isin("USD", "EUR", "GBP")) |
            (~F.col("txn_status_c").isin("SUCCESS", "FAILED", "PENDING")),
            F.lit("INVALID_DOMAIN_LABEL")
        ),
        F.when(
            F.col("order_id_c").isNotNull() & (F.col("order_id_c") != "") & F.col("_ref_order_id").isNull(),
            F.lit("ORPHAN_ORDER_REFERENCE")
        )
    )
)

txns_quarantine = txns_ruled.filter(F.col("rejection_reason") != "").select(
    "txn_id", "order_id", "payment_method", "amount", "currency", "txn_status", "txn_ts",
    "rejection_reason", "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
)
txns_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.quarantine_transactions")

w_txns = Window.partitionBy("txn_id_c").orderBy(
    F.col("txn_ts_c").desc(), F.col("_row_hash").asc(), F.col("_bronze_record_id").asc()
)

txns_valid_ranked = (
    txns_ruled.filter(F.col("rejection_reason") == "")
    .withColumn("_rn", F.row_number().over(w_txns))
    .withColumn("_winner_hash", F.first("_row_hash").over(w_txns))
)

silver_txns = txns_valid_ranked.filter(F.col("_rn") == 1).select(
    F.col("txn_id_c").alias("txn_id"),
    F.col("order_id_c").alias("order_id"),
    F.col("payment_method_c").alias("payment_method"),
    F.col("amount_c").alias("amount"),
    F.col("currency_c").alias("currency"),
    F.col("txn_status_c").alias("txn_status"),
    F.col("txn_ts_c").alias("txn_ts"),
    "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
)
silver_txns.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_transactions")

dup_txns = (
    txns_valid_ranked.filter(F.col("_rn") > 1)
    .withColumn(
        "duplicate_type",
        F.when(F.col("_row_hash") == F.col("_winner_hash"), F.lit("EXACT_ROW_DUPLICATE"))
         .otherwise(F.lit("BUSINESS_KEY_DUPLICATE"))
    )
    .select(
        F.col("txn_id_c").alias("txn_id"), "order_id", "payment_method", "amount",
        "currency", "txn_status", "txn_ts", "duplicate_type",
        "_ingested_at", "_source_file", "_batch_id", "_row_hash", "_bronze_record_id"
    )
)
dup_txns.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.duplicate_losers_transactions")
print("Silver transformation complete.")

In [0]:
%sql
-- 1. Exact Mathematical Reconciliation Proof (Bronze = Silver + Quarantine + Duplicate Losers)
SELECT
  'orders' AS dataset,
  (SELECT COUNT(*) FROM workspace.ecom_poc.bronze_orders) AS bronze_total,
  (SELECT COUNT(*) FROM workspace.ecom_poc.silver_orders) AS silver_accepted,
  (SELECT COUNT(*) FROM workspace.ecom_poc.quarantine_orders) AS quarantined,
  (SELECT COUNT(*) FROM workspace.ecom_poc.duplicate_losers_orders) AS duplicate_losers,
  (SELECT COUNT(*) FROM workspace.ecom_poc.bronze_orders) - (
    (SELECT COUNT(*) FROM workspace.ecom_poc.silver_orders) +
    (SELECT COUNT(*) FROM workspace.ecom_poc.quarantine_orders) +
    (SELECT COUNT(*) FROM workspace.ecom_poc.duplicate_losers_orders)
  ) AS discrepancy
UNION ALL
SELECT
  'transactions' AS dataset,
  (SELECT COUNT(*) FROM workspace.ecom_poc.bronze_transactions) AS bronze_total,
  (SELECT COUNT(*) FROM workspace.ecom_poc.silver_transactions) AS silver_accepted,
  (SELECT COUNT(*) FROM workspace.ecom_poc.quarantine_transactions) AS quarantined,
  (SELECT COUNT(*) FROM workspace.ecom_poc.duplicate_losers_transactions) AS duplicate_losers,
  (SELECT COUNT(*) FROM workspace.ecom_poc.bronze_transactions) - (
    (SELECT COUNT(*) FROM workspace.ecom_poc.silver_transactions) +
    (SELECT COUNT(*) FROM workspace.ecom_poc.quarantine_transactions) +
    (SELECT COUNT(*) FROM workspace.ecom_poc.duplicate_losers_transactions)
  ) AS discrepancy;






In [0]:
%sql
-- 2. Quarantine breakdown by rejection reason
SELECT 'orders' AS dataset, rejection_reason, COUNT(*) AS row_count
FROM workspace.ecom_poc.quarantine_orders GROUP BY rejection_reason
UNION ALL
SELECT 'transactions' AS dataset, rejection_reason, COUNT(*) AS row_count
FROM workspace.ecom_poc.quarantine_transactions GROUP BY rejection_reason
ORDER BY dataset, row_count DESC;

In [0]:
%sql
-- 3. Duplicate losers breakdown (exact vs key duplicates)
SELECT 'orders' AS dataset, duplicate_type, COUNT(*) AS row_count
FROM workspace.ecom_poc.duplicate_losers_orders GROUP BY duplicate_type
UNION ALL
SELECT 'transactions' AS dataset, duplicate_type, COUNT(*) AS row_count
FROM workspace.ecom_poc.duplicate_losers_transactions GROUP BY duplicate_type;

In [0]:
%sql

-- 4. Sample quarantined & duplicated records
SELECT * FROM workspace.ecom_poc.quarantine_transactions LIMIT 5;


In [0]:
%sql
SELECT * FROM workspace.ecom_poc.duplicate_losers_orders LIMIT 5;